In [0]:
from pyspark.sql.functions import current_timestamp, date_format

roll_number = "25280091"
catalog_name = f"{roll_number}-pa2"
volume_path = f"/Volumes/{catalog_name}/bronze/temp"

bronze_part2_schema_path = f"{volume_path}/schemas/bronze_flights_part2_schema"
bronze_part2_checkpoint = f"{volume_path}/checkpoints/bronze_flights_part2_cp"

flight_part2_df = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "csv")
         .option("cloudFiles.inferColumnTypes", "true")
         .option("cloudFiles.schemaLocation", bronze_part2_schema_path)
         .option("cloudFiles.schemaEvolutionMode", "rescue")
         .option("header", "true")
         .load(f"{volume_path}/flight_landing_zone_part2")
         .withColumn("ingestion_timestamp", date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
)

bronze_part2_query = (
    flight_part2_df.writeStream
                   .format("delta")
                   .outputMode("append")
                   .option("checkpointLocation", bronze_part2_checkpoint)
                   .trigger(availableNow=True)
                   .toTable(f"`{catalog_name}`.bronze.flights_part2")
)

bronze_part2_query.awaitTermination()

In [0]:
from pyspark.sql.functions import (
    col, concat_ws, lit, when, least, greatest,
    broadcast, lpad, expr
)

airport_ref_raw = (
    spark.read
         .option("header", "true")
         .option("sep", "\t")
         .csv(f"{volume_path}/flight_reference_data")
)

airport_ref = (
    airport_ref_raw
    .select(
        col("IATA").alias("iata"),
        col("City").alias("city"),
        col("State").alias("state")
    )
    .filter(col("iata").isNotNull())
)

origin_ref = (
    airport_ref
    .select(
        col("iata").alias("origin_iata"),
        col("city").alias("origin_city"),
        col("state").alias("origin_state")
    )
)

dest_ref = (
    airport_ref
    .select(
        col("iata").alias("dest_iata"),
        col("city").alias("dest_city"),
        col("state").alias("dest_state")
    )
)

bronze_part2_stream = spark.readStream.table(f"`{catalog_name}`.bronze.flights_part2")

silver_part2_df = (
    bronze_part2_stream
    .withColumn("delay", col("delay").cast("int"))
    .withColumn("distance", col("distance").cast("int"))
    .withColumn("origin", col("origin").cast("string"))
    .withColumn("destination", col("destination").cast("string"))
    .withColumn("date", col("date").cast("string"))
    .withColumn("date_padded", lpad(col("date"), 8, "0"))
    .withColumn(
        "event_timestamp",
        expr("try_to_timestamp(concat('2025', date_padded), 'yyyyMMddHHmm')")
    )
    .join(broadcast(origin_ref), col("origin") == col("origin_iata"), "left")
    .join(broadcast(dest_ref), col("destination") == col("dest_iata"), "left")
    .withColumn("departure", concat_ws(", ", col("origin_city"), col("origin_state")))
    .withColumn("arrival", concat_ws(", ", col("dest_city"), col("dest_state")))
    .filter(col("departure") != "")
    .filter(col("arrival") != "")
    .filter(col("event_timestamp").isNotNull())
    .withColumn(
        "take_off_status",
        when(col("delay") < 0, "Early")
        .when((col("delay") >= 0) & (col("delay") < 10), "On Time")
        .when((col("delay") >= 10) & (col("delay") < 30), "Late")
        .otherwise("Delay")
    )
    .withColumn(
        "standardized_route",
        concat_ws("-", least(col("origin"), col("destination")), greatest(col("origin"), col("destination")))
    )
    .withColumn("seq", lit(3))
    .select(
        "date",
        "departure",
        "arrival",
        "standardized_route",
        "distance",
        "delay",
        "take_off_status",
        "ingestion_timestamp",
        "event_timestamp",
        "seq"
    )
)

silver_part2_query = (
    silver_part2_df.writeStream
                   .format("delta")
                   .outputMode("append")
                   .option("checkpointLocation", f"{volume_path}/checkpoints/silver_flights_part2_cp")
                   .trigger(availableNow=True)
                   .toTable(f"`{catalog_name}`.silver.flights")
)

silver_part2_query.awaitTermination()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

payload_schema = StructType([
    StructField("date", StringType(), True),
    StructField("delay", IntegerType(), True),
    StructField("distance", IntegerType(), True),
    StructField("origin", StringType(), True),
    StructField("destination", StringType(), True),
    StructField("departure", StringType(), True),
    StructField("arrival", StringType(), True),
    StructField("standardized_route", StringType(), True),
    StructField("take_off_status", StringType(), True),
    StructField("ingestion_timestamp", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("seq", IntegerType(), True)
])

In [0]:
cdc_raw_df = (
    spark.readStream
         .format("json")
         .schema("id STRING, action STRING, seq INT, payload STRING")
         .load(f"{volume_path}/cdc_zone_part2")
)

In [0]:
from pyspark.sql.functions import col, from_json, row_number, to_timestamp
from pyspark.sql.window import Window
from delta.tables import DeltaTable

def merge_cdc_to_silver(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    parsed_df = (
        batch_df
        .withColumn("parsed", from_json(col("payload"), payload_schema))
        .select(
            col("id"),
            col("action"),
            col("seq"),
            col("parsed.date").alias("date"),
            col("parsed.departure").alias("departure"),
            col("parsed.arrival").alias("arrival"),
            col("parsed.standardized_route").alias("standardized_route"),
            col("parsed.distance").alias("distance"),
            col("parsed.delay").alias("delay"),
            col("parsed.take_off_status").alias("take_off_status"),
            col("parsed.ingestion_timestamp").alias("ingestion_timestamp"),
            to_timestamp(col("parsed.event_timestamp")).alias("event_timestamp")
        )
    )

    # keep highest seq only for each id inside the micro-batch
    window_spec = Window.partitionBy("id").orderBy(col("seq").desc())

    dedup_df = (
        parsed_df
        .withColumn("rn", row_number().over(window_spec))
        .filter(col("rn") == 1)
        .drop("rn")
    )

    silver_table = DeltaTable.forName(spark, f"`{catalog_name}`.silver.flights")

    (
        silver_table.alias("target")
        .merge(
            dedup_df.alias("source"),
            "target.standardized_route = source.standardized_route "
            "AND target.event_timestamp = source.event_timestamp"
        )
        .whenMatchedDelete(condition="source.action = 'delete'")
        .whenMatchedUpdate(
            condition="source.action = 'update'",
            set={
                "date": "source.date",
                "departure": "source.departure",
                "arrival": "source.arrival",
                "standardized_route": "source.standardized_route",
                "distance": "source.distance",
                "delay": "source.delay",
                "take_off_status": "source.take_off_status",
                "ingestion_timestamp": "source.ingestion_timestamp",
                "event_timestamp": "source.event_timestamp",
                "seq": "source.seq"
            }
        )
        .whenNotMatchedInsert(
            condition="source.action = 'insert'",
            values={
                "date": "source.date",
                "departure": "source.departure",
                "arrival": "source.arrival",
                "standardized_route": "source.standardized_route",
                "distance": "source.distance",
                "delay": "source.delay",
                "take_off_status": "source.take_off_status",
                "ingestion_timestamp": "source.ingestion_timestamp",
                "event_timestamp": "source.event_timestamp",
                "seq": "source.seq"
            }
        )
        .execute()
    )

In [0]:
cdc_query = (
    cdc_raw_df.writeStream
              .foreachBatch(merge_cdc_to_silver)
              .option("checkpointLocation", f"{volume_path}/checkpoints/cdc_merge_cp")
              .trigger(availableNow=True)
              .start()
)

cdc_query.awaitTermination()

In [0]:
silver_live_stream = (
    spark.readStream
         .table(f"`{catalog_name}`.silver.flights")
         .withWatermark("event_timestamp", "7 days")
)

In [0]:
from pyspark.sql.functions import (
    col, avg, round as _round,
    date_trunc, date_format, add_months
)

live_monthly_delays_df = (
    silver_live_stream
    .filter(col("event_timestamp").isNotNull())
    .groupBy(date_trunc("month", col("event_timestamp")).alias("month_start"))
    .agg(_round(avg(col("delay") / 60.0), 2).alias("moving_avg_delay"))
    .select(
        date_format(col("month_start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
        date_format(add_months(col("month_start"), 1), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
        col("moving_avg_delay")
    )
)

live_monthly_delays_query = (
    live_monthly_delays_df.writeStream
                          .format("delta")
                          .outputMode("complete")
                          .option("checkpointLocation", f"{volume_path}/checkpoints/live_gold_monthly_delays_cp")
                          .trigger(availableNow=True)
                          .toTable(f"`{catalog_name}`.gold.monthly_delays_live")
)

live_monthly_delays_query.awaitTermination()

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import current_timestamp, date_format, col, concat_ws, lit, to_timestamp, when, least, greatest, avg, sum as _sum, max as _max, expr, round as _round, window, count

roll_number = "25100225"
catalog_name = f"{roll_number}-pa2"
volume_path = f"/Volumes/{catalog_name}/bronze/temp"

@dp.table(name=f"`{catalog_name}`.bronze.flights_sdp")
def bronze_flights_sdp():
    return (
        spark.readStream
             .format("cloudFiles")
             .option("cloudFiles.format", "csv")
             .option("cloudFiles.inferColumnTypes", "true")
             .option("cloudFiles.schemaLocation", f"{volume_path}/schemas/bronze_sdp_schema")
             .option("cloudFiles.schemaEvolutionMode", "rescue")
             .option("header", "true")
             .load(f"{volume_path}/flight_landing_zone")
             .withColumn("ingestion_timestamp", date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
    )

@dp.materialized_view(name=f"`{catalog_name}`.silver.flights_sdp")
def silver_flights_sdp():
    bronze = spark.read.table(f"`{catalog_name}`.bronze.flights_sdp")
    return (
        bronze
        .withColumn("delay", col("delay").cast("int"))
        .withColumn("distance", col("distance").cast("int"))
        .withColumn("event_timestamp", to_timestamp(concat_ws("", lit("2025"), col("date")), "yyyyMMddHHmm"))
        .withColumn(
            "take_off_status",
            when(col("delay") < 0, "Early")
            .when((col("delay") >= 0) & (col("delay") < 10), "On Time")
            .when((col("delay") >= 10) & (col("delay") < 30), "Late")
            .otherwise("Delay")
        )
        .withColumn(
            "standardized_route",
            concat_ws("-", least(col("origin"), col("destination")), greatest(col("origin"), col("destination")))
        )
    )

@dp.materialized_view(name=f"`{catalog_name}`.gold.monthly_delays_sdp")
def monthly_delays_sdp():
    silver = spark.read.table(f"`{catalog_name}`.silver.flights_sdp")
    return (
        silver
        .groupBy(window(col("event_timestamp"), "1 month"))
        .agg(_round(avg(col("delay") / 60.0), 2).alias("moving_avg_delay"))
        .select(
            date_format(col("window.start"), "yyyy-MM-dd HH:mm:ss").alias("window_start"),
            date_format(col("window.end"), "yyyy-MM-dd HH:mm:ss").alias("window_end"),
            col("moving_avg_delay")
        )
    )

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File /databricks/python/lib/python3.12/site-packages/pyspark/pipelines/__init__.py:28
     27 try:
---> 28     from dlt import *
     29 except ImportError:

ModuleNotFoundError: No module named 'dlt'

During handling of the above exception, another exception occurred:

TypeError                                 Traceback (most recent call last)
File <command-7988590507226376>, line 1
----> 1 from pyspark import pipelines as dp
      2 from pyspark.sql.functions import current_timestamp, date_format, col, concat_ws, lit, to_timestamp, when, least, greatest, avg, sum as _sum, max as _max, expr, round as _round, window, count
      4 roll_number = "25100225"

File /databricks/python/lib/python3.12/site-packages/pyspark/pipelines/__init__.py:30
     28         from dlt import *
     29     except ImportError:
---> 30         raise PySparkEx

In [0]:
bronze_part2 = spark.table(f"`{catalog_name}`.bronze.flights_part2")

bronze_part2.display()
bronze_part2.printSchema()
print("Bronze Part 2 count:", bronze_part2.count())

date,delay,distance,origin,destination,_rescued_data,ingestion_timestamp
4100346,19,542,RUH,DXB,null,2026-03-28 21:04:49
4100347,133,1298,LHE,AUH,null,2026-03-28 21:04:49
4100348,251,2269,JED,LHE,null,2026-03-28 21:04:49
4100349,73,199,AUH,DOH,null,2026-03-28 21:04:49
4100350,94,2269,JED,LHE,null,2026-03-28 21:04:49
4100351,118,2269,LHE,JED,null,2026-03-28 21:04:49
4100352,69,826,JED,DOH,null,2026-03-28 21:04:49
4100353,175,307,DOH,RUH,null,2026-03-28 21:04:49
4100354,183,1579,LHE,KWI,null,2026-03-28 21:04:49
4100355,111,72,AUH,DXB,null,2026-03-28 21:04:49


root
 |-- date: integer (nullable = true)
 |-- delay: integer (nullable = true)
 |-- distance: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)

Bronze Part 2 count: 131


In [0]:
from pyspark.sql.functions import col

print("Null ingestion_timestamp:", bronze_part2.filter(col("ingestion_timestamp").isNull()).count())

Null ingestion_timestamp: 0


In [0]:
silver_df = spark.table(f"`{catalog_name}`.silver.flights")

silver_df.display()
silver_df.printSchema()
print("Silver count:", silver_df.count())

date,departure,arrival,standardized_route,distance,delay,take_off_status,ingestion_timestamp,event_timestamp,seq
1011245,"Allentown, PA","Atlanta, GA",ABE-ATL,602,6,On Time,2026-03-28 20:35:05,2025-01-01T12:45:00.000Z,3
1020600,"Allentown, PA","Detroit, MI",ABE-DTW,369,-8,Early,2026-03-28 20:35:05,2025-01-02T06:00:00.000Z,3
1021245,"Allentown, PA","Atlanta, GA",ABE-ATL,602,-2,Early,2026-03-28 20:35:05,2025-01-02T12:45:00.000Z,3
1020605,"Allentown, PA","Atlanta, GA",ABE-ATL,602,-4,Early,2026-03-28 20:35:05,2025-01-02T06:05:00.000Z,3
1031245,"Allentown, PA","Atlanta, GA",ABE-ATL,602,-4,Early,2026-03-28 20:35:05,2025-01-03T12:45:00.000Z,3
1030605,"Allentown, PA","Atlanta, GA",ABE-ATL,602,0,On Time,2026-03-28 20:35:05,2025-01-03T06:05:00.000Z,3
1041243,"Allentown, PA","Atlanta, GA",ABE-ATL,602,10,Late,2026-03-28 20:35:05,2025-01-04T12:43:00.000Z,3
1040605,"Allentown, PA","Atlanta, GA",ABE-ATL,602,28,Late,2026-03-28 20:35:05,2025-01-04T06:05:00.000Z,3
1051245,"Allentown, PA","Atlanta, GA",ABE-ATL,602,88,Delay,2026-03-28 20:35:05,2025-01-05T12:45:00.000Z,3
1050605,"Allentown, PA","Atlanta, GA",ABE-ATL,602,9,On Time,2026-03-28 20:35:05,2025-01-05T06:05:00.000Z,3


root
 |-- date: string (nullable = true)
 |-- departure: string (nullable = true)
 |-- arrival: string (nullable = true)
 |-- standardized_route: string (nullable = true)
 |-- distance: integer (nullable = true)
 |-- delay: integer (nullable = true)
 |-- take_off_status: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- seq: integer (nullable = true)

Silver count: 1361141


In [0]:
silver_df.columns

['date',
 'departure',
 'arrival',
 'standardized_route',
 'distance',
 'delay',
 'take_off_status',
 'ingestion_timestamp',
 'event_timestamp',
 'seq']

In [0]:
from pyspark.sql.functions import col

print("Null event_timestamp:", silver_df.filter(col("event_timestamp").isNull()).count())
print("Empty departure:", silver_df.filter((col("departure").isNull()) | (col("departure") == "")).count())
print("Empty arrival:", silver_df.filter((col("arrival").isNull()) | (col("arrival") == "")).count())
print("Bad seq:", silver_df.filter(col("seq").isNull()).count())

Null event_timestamp: 0
Empty departure: 0
Empty arrival: 0
Bad seq: 0


In [0]:
silver_df.groupBy("take_off_status").count().display()

take_off_status,count
On Time,336192
Early,651328
Late,181969
Delay,191652


In [0]:
silver_df.orderBy(col("event_timestamp").desc()).display()

date,departure,arrival,standardized_route,distance,delay,take_off_status,ingestion_timestamp,event_timestamp,seq
3312359,"Albuquerque, NM","New York, NY",ABQ-JFK,1586,-2,Early,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Denver, CO",Washington DC,DEN-IAD,1262,-5,Early,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Los Angeles, CA","Austin, TX",AUS-LAX,1079,-8,Early,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"San Francisco, CA","Atlanta, GA",ATL-SFO,1859,39,Delay,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Denver, CO","New York, NY",DEN-JFK,1413,-3,Early,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Anchorage, AK","Denver, CO",ANC-DEN,2090,-1,Early,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Denver, CO","Newark, NJ",DEN-EWR,1395,-5,Early,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Salt Lake City, UT","Charlotte, NC",CLT-SLC,1501,5,On Time,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Los Angeles, CA","Atlanta, GA",ATL-LAX,1691,0,On Time,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Seattle, WA","Houston, TX",IAH-SEA,1628,0,On Time,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3


In [0]:
silver_df.groupBy("seq").count().display()

seq,count
3,1361141


In [0]:
print("Bronze Part 2 count:", spark.table(f"`{catalog_name}`.bronze.flights_part2").count())
print("Silver count after CDC:", spark.table(f"`{catalog_name}`.silver.flights").count())

Bronze Part 2 count: 131
Silver count after CDC: 1361141


In [0]:
spark.sql(f"SELECT * FROM `{catalog_name}`.silver.flights ORDER BY event_timestamp DESC").display()

date,departure,arrival,standardized_route,distance,delay,take_off_status,ingestion_timestamp,event_timestamp,seq
3312359,"Albuquerque, NM","New York, NY",ABQ-JFK,1586,-2,Early,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Denver, CO",Washington DC,DEN-IAD,1262,-5,Early,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Los Angeles, CA","Austin, TX",AUS-LAX,1079,-8,Early,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"San Francisco, CA","Atlanta, GA",ATL-SFO,1859,39,Delay,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Denver, CO","New York, NY",DEN-JFK,1413,-3,Early,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Anchorage, AK","Denver, CO",ANC-DEN,2090,-1,Early,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Denver, CO","Newark, NJ",DEN-EWR,1395,-5,Early,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Salt Lake City, UT","Charlotte, NC",CLT-SLC,1501,5,On Time,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Los Angeles, CA","Atlanta, GA",ATL-LAX,1691,0,On Time,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3
3312359,"Seattle, WA","Houston, TX",IAH-SEA,1628,0,On Time,2026-03-28 20:35:05,2025-03-31T23:59:00.000Z,3


In [0]:
monthly_delays_live = spark.table(f"`{catalog_name}`.gold.monthly_delays_live")
monthly_delays_live.display()
monthly_delays_live.printSchema()
print("monthly_delays_live count:", monthly_delays_live.count())

window_start,window_end,moving_avg_delay
2025-03-01 00:00:00,2025-04-01 00:00:00,0.16
2025-01-01 00:00:00,2025-02-01 00:00:00,0.25
2025-02-01 00:00:00,2025-03-01 00:00:00,0.2


root
 |-- window_start: string (nullable = true)
 |-- window_end: string (nullable = true)
 |-- moving_avg_delay: double (nullable = true)

monthly_delays_live count: 3


In [0]:
spark.sql(f"SHOW TABLES IN `{catalog_name}`.bronze").display()
spark.sql(f"SHOW TABLES IN `{catalog_name}`.silver").display()
spark.sql(f"SHOW TABLES IN `{catalog_name}`.gold").display()

database,tableName,isTemporary
bronze,flights,false
bronze,flights_part2,false


database,tableName,isTemporary
silver,flights,false


database,tableName,isTemporary
gold,biweekly_distance,false
gold,hourly_flight_counts,false
gold,monthly_delays,false
gold,monthly_delays_live,false
gold,monthly_max_delay,false
gold,route_sliding_avg_delay,false
gold,status_sliding_counts,false
gold,weekly_route_distance,false


In [0]:
print("bronze.flights:", spark.table(f"`{catalog_name}`.bronze.flights").count())
print("bronze.flights_part2:", spark.table(f"`{catalog_name}`.bronze.flights_part2").count())
print("silver.flights:", spark.table(f"`{catalog_name}`.silver.flights").count())

print("gold.monthly_delays:", spark.table(f"`{catalog_name}`.gold.monthly_delays").count())
print("gold.biweekly_distance:", spark.table(f"`{catalog_name}`.gold.biweekly_distance").count())
print("gold.monthly_max_delay:", spark.table(f"`{catalog_name}`.gold.monthly_max_delay").count())
print("gold.hourly_flight_counts:", spark.table(f"`{catalog_name}`.gold.hourly_flight_counts").count())
print("gold.monthly_delays_live:", spark.table(f"`{catalog_name}`.gold.monthly_delays_live").count())
print("gold.route_sliding_avg_delay:", spark.table(f"`{catalog_name}`.gold.route_sliding_avg_delay").count())
print("gold.status_sliding_counts:", spark.table(f"`{catalog_name}`.gold.status_sliding_counts").count())
print("gold.weekly_route_distance:", spark.table(f"`{catalog_name}`.gold.weekly_route_distance").count())

bronze.flights: 1391578
bronze.flights_part2: 131
silver.flights: 1361141
gold.monthly_delays: 3
gold.biweekly_distance: 8
gold.monthly_max_delay: 3
gold.hourly_flight_counts: 1979
gold.monthly_delays_live: 3
gold.route_sliding_avg_delay: 183297
gold.status_sliding_counts: 740
gold.weekly_route_distance: 26651


In [0]:
spark.table(f"`{catalog_name}`.silver.flights").display()
spark.table(f"`{catalog_name}`.gold.monthly_delays_live").display()

date,departure,arrival,standardized_route,distance,delay,take_off_status,ingestion_timestamp,event_timestamp,seq
1011245,"Allentown, PA","Atlanta, GA",ABE-ATL,602,6,On Time,2026-03-28 20:35:05,2025-01-01T12:45:00.000Z,3
1020600,"Allentown, PA","Detroit, MI",ABE-DTW,369,-8,Early,2026-03-28 20:35:05,2025-01-02T06:00:00.000Z,3
1021245,"Allentown, PA","Atlanta, GA",ABE-ATL,602,-2,Early,2026-03-28 20:35:05,2025-01-02T12:45:00.000Z,3
1020605,"Allentown, PA","Atlanta, GA",ABE-ATL,602,-4,Early,2026-03-28 20:35:05,2025-01-02T06:05:00.000Z,3
1031245,"Allentown, PA","Atlanta, GA",ABE-ATL,602,-4,Early,2026-03-28 20:35:05,2025-01-03T12:45:00.000Z,3
1030605,"Allentown, PA","Atlanta, GA",ABE-ATL,602,0,On Time,2026-03-28 20:35:05,2025-01-03T06:05:00.000Z,3
1041243,"Allentown, PA","Atlanta, GA",ABE-ATL,602,10,Late,2026-03-28 20:35:05,2025-01-04T12:43:00.000Z,3
1040605,"Allentown, PA","Atlanta, GA",ABE-ATL,602,28,Late,2026-03-28 20:35:05,2025-01-04T06:05:00.000Z,3
1051245,"Allentown, PA","Atlanta, GA",ABE-ATL,602,88,Delay,2026-03-28 20:35:05,2025-01-05T12:45:00.000Z,3
1050605,"Allentown, PA","Atlanta, GA",ABE-ATL,602,9,On Time,2026-03-28 20:35:05,2025-01-05T06:05:00.000Z,3


window_start,window_end,moving_avg_delay
2025-03-01 00:00:00,2025-04-01 00:00:00,0.16
2025-01-01 00:00:00,2025-02-01 00:00:00,0.25
2025-02-01 00:00:00,2025-03-01 00:00:00,0.2
